In [1]:
!pip install -q tensorboardX scipy scikit-learn opencv-python-headless tqdm

In [2]:
!pip uninstall -y -q imageio-ffmpeg
!pip install -q "imageio-ffmpeg==0.4.5"

In [3]:
import imageio_ffmpeg, os

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg binary tại:', ffmpeg_path)

os.makedirs('/usr/local/bin', exist_ok=True)
symlink_path = '/usr/local/bin/ffmpeg'
if os.path.exists(symlink_path) or os.path.islink(symlink_path):
    os.remove(symlink_path)
os.symlink(ffmpeg_path, symlink_path)

import subprocess
result = subprocess.run(['ffmpeg', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print('ffmpeg return code:', result.returncode)
print(result.stdout.decode()[:200])

ffmpeg binary tại: /opt/conda/lib/python3.6/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux64-v4.2.2
ffmpeg return code: 0
ffmpeg version 4.2.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2019 the FFmpeg developers
built with gcc 8 (Debian 8.3.0-6)
configuration: --enable-gpl --enable-version3 --enable-st


In [4]:
import os
import sys
import copy
import glob
import time
import math
import random
import numbers
import pickle
import shutil
import argparse
import re
import csv
import subprocess
import collections.abc as collections

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
plt.switch_backend('agg')
from collections import deque
from datetime import datetime

from PIL import ImageOps, Image
from scipy.io import wavfile
from scipy.interpolate import interp1d
from scipy import signal
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils import data
import torch.utils.data
import torchvision
from torchvision import transforms
import torchvision.transforms.functional as TF
from tensorboardX import SummaryWriter
from tqdm import tqdm

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch version: 1.3.0
CUDA available: True
GPU: Tesla T4


---
## 2. Configuration

In [5]:
class Config:
    # Data paths - Kaggle mounts DFDC data here
    DFDC_DATA_DIR = '/kaggle/input/competitions/deepfake-detection-challenge'   
    
    # Working directory for preprocessed data
    WORK_DIR = '/kaggle/working'
    TRAIN_DIR = os.path.join(WORK_DIR, 'train')
    TEST_DIR = os.path.join(WORK_DIR, 'test')
    
    # Training hyperparameters
    BATCH_SIZE = 16
    NUM_WORKERS = 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    EPOCHS = 100
    IMG_DIM = 224
    SPATIAL_SIZE = 28
    NET = 'resnet18'
    
    # Model options
    WITH_ATTENTION = True
    RESIDUAL_CONN = False
    USING_PSEUDO_FAKE = True
    
    # Pseudo-fake augmentation parameters
    AUD_MIN_FAKE_LEN = 2
    AUD_MAX_FAKE_LEN = -1
    VIS_MIN_FAKE_LEN = 2
    VIS_MAX_FAKE_LEN = -1
    
    # Misc
    PRINT_FREQ = 10
    SEED = 0
    SAVE_ALL = False

cfg = Config()
print('Configuration loaded.')
print(f'  DFDC Data: {cfg.DFDC_DATA_DIR}')
print(f'  Working Dir: {cfg.WORK_DIR}')
print(f'  Epochs: {cfg.EPOCHS}, BS: {cfg.BATCH_SIZE}, LR: {cfg.LEARNING_RATE}')
print(f'  Attention: {cfg.WITH_ATTENTION}, Pseudo-Fake: {cfg.USING_PSEUDO_FAKE}')

Configuration loaded.
  DFDC Data: /kaggle/input/competitions/deepfake-detection-challenge
  Working Dir: /kaggle/working
  Epochs: 100, BS: 16, LR: 0.0001
  Attention: True, Pseudo-Fake: True


---
## 3. Utility Functions

In [6]:
def min_max_normalize(value, min_value, max_value):
    return (value - min_value) / (max_value - min_value + 0.00000001)


def save_checkpoint(state, is_best=0, gap=1, filename='models/checkpoint.pth.tar', keep_all=False):
    torch.save(state, filename)
    last_epoch_path = os.path.join(os.path.dirname(filename),
                                   'epoch%s.pth.tar' % str(state['epoch'] - gap))
    if (state['epoch'] - gap) == 50:
        os.makedirs(os.path.join(os.path.dirname(filename), 'halfepochs_results'), exist_ok=True)
        if os.path.exists(last_epoch_path):
            os.rename(last_epoch_path,
                      os.path.join(os.path.dirname(filename), 'halfepochs_results',
                                   'epoch%s.pth.tar' % str(state['epoch'] - gap)))
        past_best = glob.glob(os.path.join(os.path.dirname(filename), 'model_best_*.pth.tar'))
        for i in past_best:
            os.rename(i, os.path.join(os.path.dirname(filename), 'halfepochs_results', os.path.basename(i)))
    if not keep_all:
        try:
            os.remove(last_epoch_path)
        except:
            pass
    if is_best:
        past_best = glob.glob(os.path.join(os.path.dirname(filename), 'model_best_*.pth.tar'))
        for i in past_best:
            try:
                os.remove(i)
            except:
                pass
        torch.save(state, os.path.join(os.path.dirname(filename),
                                       'model_best_epoch%s.pth.tar' % str(state['epoch'])))


def write_log(content, epoch, filename):
    if not os.path.exists(filename):
        log_file = open(filename, 'w')
    else:
        log_file = open(filename, 'a')
    log_file.write('## Epoch %d:\n' % epoch)
    log_file.write('time: %s\n' % str(datetime.now()))
    log_file.write(content + '\n\n')
    log_file.close()


def denorm(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    assert len(mean) == len(std) == 3
    inv_mean = [-mean[i] / std[i] for i in range(3)]
    inv_std = [1 / i for i in std]
    return transforms.Normalize(mean=inv_mean, std=inv_std)


class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.local_history = deque([])
        self.local_avg = 0
        self.history = []
        self.dict = {}
        self.save_dict = {}

    def update(self, val, n=1, history=0, step=5):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        if history:
            self.history.append(val)
        if step > 0:
            self.local_history.append(val)
            if len(self.local_history) > step:
                self.local_history.popleft()
            self.local_avg = np.average(self.local_history)

    def dict_update(self, val, key):
        if key in self.dict.keys():
            self.dict[key].append(val)
        else:
            self.dict[key] = [val]

    def __len__(self):
        return self.count

print('Utility functions defined.')

Utility functions defined.


---
## 4. Augmentation Transforms

In [7]:
class Padding:
    def __init__(self, pad):
        self.pad = pad
    def __call__(self, img):
        return ImageOps.expand(img, border=self.pad, fill=0)


class Scale:
    def __init__(self, size, interpolation=Image.NEAREST):
        assert isinstance(size, int) or (isinstance(size, collections.Iterable) and len(size) == 2)
        self.size = size
        self.interpolation = interpolation

    def __call__(self, imgmap):
        img1 = imgmap[0]
        if isinstance(self.size, int):
            w, h = img1.size
            if (w <= h and w == self.size) or (h <= w and h == self.size):
                return imgmap
            if w < h:
                ow = self.size
                oh = int(self.size * h / w)
                return [i.resize((ow, oh), self.interpolation) for i in imgmap]
            else:
                oh = self.size
                ow = int(self.size * w / h)
                return [i.resize((ow, oh), self.interpolation) for i in imgmap]
        else:
            return [i.resize(self.size, self.interpolation) for i in imgmap]


class CenterCrop:
    def __init__(self, size, consistent=True):
        if isinstance(size, numbers.Number):
            self.size = (int(size), int(size))
        else:
            self.size = size
    def __call__(self, imgmap):
        img1 = imgmap[0]
        w, h = img1.size
        th, tw = self.size
        x1 = int(round((w - tw) / 2.))
        y1 = int(round((h - th) / 2.))
        return [i.crop((x1, y1, x1 + tw, y1 + th)) for i in imgmap]


class RandomHorizontalFlip:
    def __init__(self, consistent=True, command=None):
        self.consistent = consistent
        if command == 'left':
            self.threshold = 0
        elif command == 'right':
            self.threshold = 1
        else:
            self.threshold = 0.5
    def __call__(self, imgmap):
        if self.consistent:
            if random.random() < self.threshold:
                return [i.transpose(Image.FLIP_LEFT_RIGHT) for i in imgmap]
            else:
                return imgmap
        else:
            result = []
            for i in imgmap:
                if random.random() < self.threshold:
                    result.append(i.transpose(Image.FLIP_LEFT_RIGHT))
                else:
                    result.append(i)
            return result


class ToTensor:
    def __call__(self, imgmap):
        totensor = transforms.ToTensor()
        return [totensor(i) for i in imgmap]


class Normalize:
    def __init__(self, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
        self.mean = mean
        self.std = std
    def __call__(self, imgmap):
        normalize = transforms.Normalize(mean=self.mean, std=self.std)
        return [normalize(i) for i in imgmap]

print('Augmentation transforms defined.')

Augmentation transforms defined.


---
## 5. ResNet 2D3D Backbone

In [8]:
def conv3x3x3(in_planes, out_planes, stride=1, bias=False):
    return nn.Conv3d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=bias)

def conv1x3x3(in_planes, out_planes, stride=1, bias=False):
    return nn.Conv3d(in_planes, out_planes, kernel_size=(1, 3, 3),
                     stride=(1, stride, stride), padding=(0, 1, 1), bias=bias)


def downsample_basic_block(x, planes, stride):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.Tensor(
        out.size(0), planes - out.size(1), out.size(2), out.size(3),
        out.size(4)).zero_()
    if isinstance(out.data, torch.cuda.FloatTensor):
        zero_pads = zero_pads.cuda()
    out = Variable(torch.cat([out.data, zero_pads], dim=1))
    return out


class BasicBlock3d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 track_running_stats=True, use_final_relu=True):
        super(BasicBlock3d, self).__init__()
        bias = False
        self.use_final_relu = use_final_relu
        self.conv1 = conv3x3x3(inplanes, planes, stride, bias=bias)
        self.bn1 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, bias=bias)
        self.bn2 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        if self.use_final_relu:
            out = self.relu(out)
        return out


class BasicBlock2d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 track_running_stats=True, use_final_relu=True):
        super(BasicBlock2d, self).__init__()
        bias = False
        self.use_final_relu = use_final_relu
        self.conv1 = conv1x3x3(inplanes, planes, stride, bias=bias)
        self.bn1 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1x3x3(planes, planes, bias=bias)
        self.bn2 = nn.BatchNorm3d(planes, track_running_stats=track_running_stats)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        if self.use_final_relu:
            out = self.relu(out)
        return out


class ResNet2d3d_half(nn.Module):
    def __init__(self, block, layers, track_running_stats=True):
        super(ResNet2d3d_half, self).__init__()
        self.inplanes = 64
        self.track_running_stats = track_running_stats
        bias = False
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2),
                               padding=(0, 3, 3), bias=bias)
        self.bn1 = nn.BatchNorm3d(64, track_running_stats=track_running_stats)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))

        if not isinstance(block, list):
            block = [block] * 4

        self.layer1 = self._make_layer(block[0], 64, layers[0])
        self.layer2 = self._make_layer(block[1], 128, layers[1], stride=2, is_final=True)

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                m.weight = nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1, is_final=False):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if block == BasicBlock2d:
                customized_stride = (1, stride, stride)
            else:
                customized_stride = stride
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=customized_stride, bias=False),
                nn.BatchNorm3d(planes * block.expansion,
                               track_running_stats=self.track_running_stats)
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample,
                            track_running_stats=self.track_running_stats))
        self.inplanes = planes * block.expansion
        if is_final:
            for i in range(1, blocks - 1):
                layers.append(block(self.inplanes, planes,
                                    track_running_stats=self.track_running_stats))
            layers.append(block(self.inplanes, planes,
                                track_running_stats=self.track_running_stats,
                                use_final_relu=False))
        else:
            for i in range(1, blocks):
                layers.append(block(self.inplanes, planes,
                                    track_running_stats=self.track_running_stats))
        return nn.Sequential(*layers)

    def forward(self, x, return_intermediate=False):
        x0 = self.conv1(x)
        x0 = self.bn1(x0)
        x0 = self.relu(x0)
        x0 = self.maxpool(x0)
        x1 = self.layer1(x0)
        x = self.layer2(x1)
        if return_intermediate:
            return x, x1
        else:
            return x


def resnet18_2d3d_half(**kwargs):
    model = ResNet2d3d_half([BasicBlock3d, BasicBlock3d], [2, 2], **kwargs)
    return model


def resnet34_2d3d_half(**kwargs):
    model = ResNet2d3d_half([BasicBlock3d, BasicBlock3d], [6, 3], **kwargs)
    return model


def select_resnet_half(network, track_running_stats=True):
    param = {'feature_size': 512}
    if network == 'resnet18':
        model = resnet18_2d3d_half(track_running_stats=track_running_stats)
        param['feature_size'] = 128
    elif network == 'resnet34':
        model = resnet34_2d3d_half(track_running_stats=track_running_stats)
        param['feature_size'] = 128
    else:
        raise IOError('model type is wrong')
    return model, param


def neq_load_customized(model, pretrained_dict):
    model_dict = model.state_dict()
    tmp = {}
    print('\n=======Check Weights Loading======')
    print('Weights not used from pretrained file:')
    for k, v in pretrained_dict.items():
        if k in model_dict:
            tmp[k] = v
        else:
            print(k)
    print('---------------------------')
    print('Weights not loaded into new model:')
    for k, v in model_dict.items():
        if k not in pretrained_dict:
            print(k)
    print('===================================\n')
    del pretrained_dict
    model_dict.update(tmp)
    del tmp
    model.load_state_dict(model_dict)
    return model

print('ResNet 2D3D backbone defined.')

ResNet 2D3D backbone defined.


---
## 6. FGI Model (Audio-Visual Network)

In [9]:
class My_CNN_RawAud(nn.Module):
    def __init__(self):
        super(My_CNN_RawAud, self).__init__()
        self.netcnnaud_layer1 = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=80, stride=8),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
        )
        self.netcnnaud_layer2 = nn.Sequential(
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Conv1d(128, 192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(192),
            nn.ReLU(inplace=True),
            nn.Conv1d(192, 192, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(192),
        )

    def forward(self, x):
        x1 = self.netcnnaud_layer1(x)
        x2 = self.netcnnaud_layer2(x1)
        return x2


class My_Network(nn.Module):
    def __init__(self, network='resnet18', with_attention=False,
                 residual_conn=False, spatial_size=28):
        super(My_Network, self).__init__()
        self.__nFeatures__ = 24
        self.__nChs__ = 32
        self.__midChs__ = 32
        self.aud2visspace = None
        self.with_attention = with_attention
        self.residual_conn = residual_conn
        self.spatial_size = spatial_size

        self.netcnnaud = My_CNN_RawAud()

        a_size = [192, 1497]
        v_size = [128, 15, spatial_size, spatial_size]
        stride = int(a_size[1] / v_size[1])
        kernel_size = int(a_size[1] - (v_size[1] - 1) * stride)
        self.netcnnaud_to_vis = nn.Sequential(
            nn.Conv1d(a_size[0], v_size[0] // 2, kernel_size=kernel_size, stride=stride),
            nn.BatchNorm1d(v_size[0] // 2),
            nn.ReLU(),
            nn.Conv1d(v_size[0] // 2, v_size[0], kernel_size=3, stride=1, padding=1)
        )

        self.netcnnlip, self.param = select_resnet_half(network, track_running_stats=False)

        map_size = v_size[2] * v_size[3]
        self.final_fc = nn.Sequential(
            nn.Linear(map_size, 1),
            nn.Sigmoid()
        )

        out_emb = v_size[0] // 4
        self.img_emb_layer = nn.Conv3d(v_size[0], out_emb, 1)
        self.aud_emb_layer = nn.Conv1d(v_size[0], out_emb, 1)

    def forward_aud(self, x):
        (B, C) = x.shape
        x = x.view(B, 1, C)
        x = self.netcnnaud(x)
        x = self.netcnnaud_to_vis(x)
        return x

    def forward_lip(self, x):
        (B, N, C, NF, H, W) = x.shape
        x = x.view(B * N, C, NF, H, W)
        x = self.netcnnlip(x, return_intermediate=False)
        if self.spatial_size != 28:
            x = nn.functional.adaptive_avg_pool3d(x, (15, self.spatial_size, self.spatial_size))
        return x

    def forward(self, vid_seq, aud_seq):
        vid_out = self.forward_lip(vid_seq)
        aud_out = self.forward_aud(aud_seq)

        aud_out = aud_out.view(aud_out.shape[0], aud_out.shape[1],
                               aud_out.shape[2], 1, 1)

        vid_aud_distance_ = torch.pow((vid_out - aud_out), 2)
        vid_aud_distance_ = vid_aud_distance_.view(
            vid_aud_distance_.shape[0],
            vid_aud_distance_.shape[1] * vid_aud_distance_.shape[2],
            vid_aud_distance_.shape[3] * vid_aud_distance_.shape[4]
        )
        vid_aud_distance_ = torch.sqrt(torch.sum(vid_aud_distance_, dim=1))

        if self.with_attention:
            img_emb = self.img_emb_layer(vid_out)
            aud_emb = self.aud_emb_layer(
                aud_out.view(aud_out.shape[0], aud_out.shape[1], aud_out.shape[2])
            )
            atts = []
            for i_emb, a_emb in zip(img_emb, aud_emb):
                atts.append(torch.tensordot(i_emb, a_emb, dims=([0, 1], [0, 1])))
            att = torch.stack(atts)
            att = att / (32 * 15)
            att = att.view(att.shape[0], -1)
            att = torch.nn.functional.softmax(att, dim=1)

            if self.residual_conn:
                vid_aud_distance_ = torch.mul(att, vid_aud_distance_) + vid_aud_distance_
            else:
                vid_aud_distance_ = torch.mul(att, vid_aud_distance_)
        else:
            att = None

        final_out = self.final_fc(vid_aud_distance_)
        return final_out, vid_aud_distance_, att


print('FGI Model defined.')
_model = My_Network(with_attention=True)
total_params = sum(p.numel() for p in _model.parameters())
print(f'Total parameters: {total_params:,}')
del _model

FGI Model defined.
Total parameters: 3,604,177


---
## 7. Dataset

In [10]:
def pil_loader(path):
    with open(path, 'rb') as f:
        with Image.open(f) as img:
            return img.convert('RGB')


def my_collate_rawaudio(batch):
    batch = list(filter(lambda x: x is not None and x[1].size()[0] == 48000, batch))
    if len(batch) == 0:
        return [[], [], [], [], [], [], []]
    return torch.utils.data.dataloader.default_collate(batch)


class deepfake_3d_rawaudio(data.Dataset):
    def __init__(self, out_dir, mode='train', transform=None,
                 vis_min_fake_len=2, vis_max_fake_len=-1,
                 aud_min_fake_len=2, aud_max_fake_len=-1,
                 using_pseudo_fake=False, dataset_name='dfdc'):
        assert dataset_name in ['dfdc', 'fakeavceleb']
        self.mode = mode
        self.transform = transform
        self.out_dir = out_dir
        self.dataset_name = dataset_name

        if mode == 'train':
            split = os.path.join(self.out_dir, 'train_split.csv')
            video_info = pd.read_csv(split, header=None)
        elif mode == 'test':
            split = os.path.join(self.out_dir, 'test_imbalance_split.csv')
            video_info = pd.read_csv(split, header=None)
        else:
            raise ValueError('wrong mode')

        self.using_pseudo_fake = using_pseudo_fake if mode == 'train' else False

        self.label_dict_encode = {'fake': 0, 'real': 1}
        self.label_dict_decode = {'0': 'fake', '1': 'real'}

        self.video_info = video_info
        self.vis_min_fake_len = vis_min_fake_len
        self.vis_max_fake_len = vis_max_fake_len
        self.aud_min_fake_len = aud_min_fake_len
        self.aud_max_fake_len = aud_max_fake_len

    def _generate_pseudo_fake(self, t_seq, audio, other_t_seq, other_audio):
        chosen_type = random.choice([0, 1, 2])
        if chosen_type == 0:
            audio = self._augment_pseudo_fake(audio, audio.shape[0],
                                              minimum_fake_length=self.aud_min_fake_len,
                                              maximum_fake_length=self.aud_max_fake_len,
                                              other_data=other_audio)
        elif chosen_type == 1:
            t_seq = self._augment_pseudo_fake(t_seq, t_seq.shape[0],
                                              minimum_fake_length=self.vis_min_fake_len,
                                              maximum_fake_length=self.vis_max_fake_len,
                                              other_data=other_t_seq)
        elif chosen_type == 2:
            audio = self._augment_pseudo_fake(audio, audio.shape[0],
                                              minimum_fake_length=self.aud_min_fake_len,
                                              maximum_fake_length=self.aud_max_fake_len,
                                              other_data=other_audio)
            t_seq = self._augment_pseudo_fake(t_seq, t_seq.shape[0],
                                              minimum_fake_length=self.vis_min_fake_len,
                                              maximum_fake_length=self.vis_max_fake_len,
                                              other_data=other_t_seq)
        return t_seq, audio, chosen_type

    def _select_pseudo_fake_window(self, data_length, minimum=2, maximum=-1):
        if 0 < minimum <= 1:
            minimum = max(2, int(minimum * data_length))
        if minimum == -1:
            minimum = data_length
        if maximum == -1:
            maximum = data_length
        elif 0 < maximum <= 1:
            maximum = min(minimum, int(maximum * data_length))
        assert minimum >= 2
        assert maximum >= minimum
        fake_len = random.randint(minimum, maximum)
        start_pos = random.randint(0, data_length - fake_len)
        end_pos = min(data_length, start_pos + fake_len)
        return fake_len, start_pos, end_pos

    def _replace_with_other(self, data_tensor, fake_len, start_pos, end_pos, other_data):
        data_tensor[start_pos:end_pos] = other_data[start_pos:end_pos]
        return data_tensor

    def _augment_pseudo_fake(self, data_tensor, time_len,
                             minimum_fake_length=2, maximum_fake_length=-1,
                             other_data=None):
        fake_len, start_pos, end_pos = self._select_pseudo_fake_window(
            data_length=time_len, minimum=minimum_fake_length, maximum=maximum_fake_length)
        data_tensor = self._replace_with_other(data_tensor, fake_len, start_pos, end_pos, other_data)
        return data_tensor

    def _get_other_item(self, index):
        if self.using_pseudo_fake and self.mode == 'train':
            success = 0
            while success == 0:
                try:
                    other_index = random.randint(0, self.__len__() - 1)
                    while index == other_index:
                        other_index = random.randint(0, self.__len__() - 1)
                    other_vpath, other_audiopath, other_label = self.video_info.iloc[other_index]
                    other_vpath = os.path.join(self.out_dir, other_vpath)
                    other_audiopath = os.path.join(self.out_dir, other_audiopath)
                    other_seq = [pil_loader(os.path.join(other_vpath, img)) for img in
                                 sorted(os.listdir(other_vpath))]
                    other_sample_rate, other_audio = wavfile.read(other_audiopath)
                    other_t_seq = self.transform(other_seq)
                    other_t_seq = torch.stack(other_t_seq, 0)
                    other_normalized_raw_audio = min_max_normalize(
                        other_audio, int(other_audio.min()), int(other_audio.max()))
                    other_normalized_raw_audio = torch.from_numpy(
                        other_normalized_raw_audio.astype(float)).float()
                    other_normalized_raw_audio = (other_normalized_raw_audio - 0.5) / 0.5
                    if len(other_normalized_raw_audio) < 48000 or len(other_t_seq) < 30:
                        continue
                    success = 1
                except:
                    continue
        else:
            other_t_seq = None
            other_normalized_raw_audio = None
        return other_t_seq, other_normalized_raw_audio

    def __getitem__(self, index):
        success = 0
        while success == 0:
            try:
                vpath, audiopath, label = self.video_info.iloc[index]
                vpath = os.path.join(self.out_dir, vpath)
                audiopath = os.path.join(self.out_dir, audiopath)
                seq = [pil_loader(os.path.join(vpath, img)) for img in sorted(os.listdir(vpath))]
                sample_rate, audio = wavfile.read(audiopath)
                t_seq = self.transform(seq)
                (C, H, W) = t_seq[0].size()
                t_seq = torch.stack(t_seq, 0)
                normalized_raw_audio = min_max_normalize(audio, int(audio.min()), int(audio.max()))
                normalized_raw_audio = torch.from_numpy(normalized_raw_audio.astype(float)).float()
                normalized_raw_audio = (normalized_raw_audio - 0.5) / 0.5
                success = 1
            except:
                index = random.randint(0, self.__len__() - 1)
                continue

        if self.using_pseudo_fake:
            use_pseudo_fake = random.randint(0, 1)
            if use_pseudo_fake:
                other_t_seq, other_normalized_raw_audio = self._get_other_item(index)
                t_seq, normalized_raw_audio, chosen_type = self._generate_pseudo_fake(
                    t_seq, normalized_raw_audio, other_t_seq, other_normalized_raw_audio)
                label = 'fake'

        t_seq = t_seq.view(1, 30, C, H, W).transpose(1, 2)
        vid = self.label_dict_encode[label]
        return t_seq, normalized_raw_audio, torch.LongTensor([vid]), audiopath

    def __len__(self):
        return len(self.video_info)

    def encode_label(self, label_name):
        return self.label_dict_encode[label_name]

print('Dataset class defined.')

Dataset class defined.


---
## 8. Preprocessing: Extract Frames and Audio from DFDC Videos

This section processes raw DFDC videos:
1. Converts videos to 30fps
2. Extracts frames as JPG images (resized to 224x224)
3. Extracts audio as WAV (48kHz, mono, PCM)
4. Splits into 1-second chunks (30 frames + corresponding audio)

> **Note**: Face detection (S3FD) is skipped. Uses full frame (similar to `--dont_crop_face`).

In [11]:
def preprocess_video_simple(video_path, output_base_dir, label, max_chunks=None):
    """
    Simplified preprocessing for a single video.
    """
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    pytmp_dir = os.path.join(output_base_dir, 'pytmp', label, video_name)
    
    if os.path.exists(pytmp_dir) and len(os.listdir(pytmp_dir)) > 0:
        return
    
    temp_frames = os.path.join(output_base_dir, 'temp_frames', video_name)
    temp_audio = os.path.join(output_base_dir, 'temp_audio')
    os.makedirs(temp_frames, exist_ok=True)
    os.makedirs(temp_audio, exist_ok=True)
    os.makedirs(pytmp_dir, exist_ok=True)
    
    try:
        cmd_frames = (
            f'ffmpeg -y -i "{video_path}" -r 30 -qscale:v 2 '
            f'-vf scale=224:224 -threads 1 -f image2 '
            f'"{os.path.join(temp_frames, "%06d.jpg")}"'
        )
        subprocess.call(cmd_frames, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        audio_path = os.path.join(temp_audio, f'{video_name}.wav')
        cmd_audio = (
            f'ffmpeg -y -i "{video_path}" -ac 1 -vn '
            f'-acodec pcm_s16le -ar 48000 "{audio_path}"'
        )
        subprocess.call(cmd_audio, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        if not os.path.exists(audio_path):
            shutil.rmtree(pytmp_dir, ignore_errors=True)
            return
        
        total_frames = len([f for f in os.listdir(temp_frames) if f.endswith('.jpg')])
        chunk_count = 0
        
        for frame_start in range(0, total_frames, 30):
            if frame_start + 30 > total_frames:
                continue
            if max_chunks and chunk_count >= max_chunks:
                break
            
            chunk_id = '%05d' % (frame_start // 30)
            chunk_dir = os.path.join(pytmp_dir, chunk_id)
            os.makedirs(chunk_dir, exist_ok=True)
            
            for i in range(frame_start + 1, frame_start + 31):
                src = os.path.join(temp_frames, '%06d.jpg' % i)
                if os.path.exists(src):
                    shutil.copy(src, chunk_dir)
            
            chunk_audio = os.path.join(pytmp_dir, f'{chunk_id}.wav')
            audio_start = frame_start / 30.0
            audio_end = (frame_start + 30) / 30.0
            cmd_chunk_audio = (
                f'ffmpeg -y -i "{audio_path}" -ss {audio_start:.3f} '
                f'-to {audio_end:.3f} "{chunk_audio}"'
            )
            subprocess.call(cmd_chunk_audio, shell=True,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            chunk_count += 1
        
        shutil.rmtree(temp_frames, ignore_errors=True)
        if os.path.exists(audio_path):
            os.remove(audio_path)
            
    except Exception as e:
        print(f'Error processing {video_name}: {e}')
        shutil.rmtree(pytmp_dir, ignore_errors=True)
        shutil.rmtree(temp_frames, ignore_errors=True)

print('Preprocessing functions defined.')

Preprocessing functions defined.


In [12]:
import json

def organize_dfdc_data(dfdc_dir, work_dir, max_videos_per_class=None):
    """
    Organize DFDC data into real/fake folders and preprocess.
    Supports both train_sample_videos and full dfdc_train_part_XX format.
    """
    train_dir = os.path.join(work_dir, 'train')
    
    metadata_path = os.path.join(dfdc_dir, 'metadata.json')
    if not os.path.exists(metadata_path):
        metadata_path = os.path.join(dfdc_dir, 'train_sample_videos', 'metadata.json')
    
    if not os.path.exists(metadata_path):
        part_dirs = sorted(glob.glob(os.path.join(dfdc_dir, 'dfdc_train_part_*')))
        if not part_dirs:
            print(f'ERROR: Cannot find DFDC data in {dfdc_dir}')
            print(f'Contents of {dfdc_dir}:')
            if os.path.exists(dfdc_dir):
                for item in os.listdir(dfdc_dir)[:20]:
                    print(f'  {item}')
            return
        
        real_count = 0
        fake_count = 0
        for part_dir in part_dirs:
            part_metadata = os.path.join(part_dir, 'metadata.json')
            if not os.path.exists(part_metadata):
                continue
            with open(part_metadata, 'r') as f:
                metadata = json.load(f)
            
            for video_name, info in metadata.items():
                video_path = os.path.join(part_dir, video_name)
                if not os.path.exists(video_path):
                    continue
                label = 'real' if info['label'] == 'REAL' else 'fake'
                
                if max_videos_per_class:
                    if label == 'real' and real_count >= max_videos_per_class:
                        continue
                    if label == 'fake' and fake_count >= max_videos_per_class:
                        continue
                
                preprocess_video_simple(video_path, train_dir, label, max_chunks=5)
                
                if label == 'real':
                    real_count += 1
                else:
                    fake_count += 1
                
                if (real_count + fake_count) % 50 == 0:
                    print(f'Processed {real_count} real, {fake_count} fake videos...')
        
        print(f'Total processed: {real_count} real, {fake_count} fake videos')
        return
    
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    video_dir = os.path.dirname(metadata_path)
    real_count = 0
    fake_count = 0
    
    for video_name, info in tqdm(metadata.items(), desc='Preprocessing videos'):
        video_path = os.path.join(video_dir, video_name)
        if not os.path.exists(video_path):
            continue
        label = 'real' if info['label'] == 'REAL' else 'fake'
        
        if max_videos_per_class:
            if label == 'real' and real_count >= max_videos_per_class:
                continue
            if label == 'fake' and fake_count >= max_videos_per_class:
                continue
        
        preprocess_video_simple(video_path, train_dir, label, max_chunks=5)
        
        if label == 'real':
            real_count += 1
        else:
            fake_count += 1
    
    print(f'\nTotal processed: {real_count} real, {fake_count} fake videos')

print('DFDC organizer defined.')

DFDC organizer defined.


In [13]:
# ====================== RUN PREPROCESSING ======================
# Set MAX_VIDEOS = None for full dataset, or a small number for testing

print('Starting DFDC data preprocessing...')
print(f'Input directory: {cfg.DFDC_DATA_DIR}')
print(f'Output directory: {cfg.WORK_DIR}')

# Adjust this based on your needs:
#   None  -> Process ALL videos (long time)
#   50    -> Quick test
#   500   -> Medium experiment
MAX_VIDEOS = 300

organize_dfdc_data(cfg.DFDC_DATA_DIR, cfg.WORK_DIR, max_videos_per_class=MAX_VIDEOS)

for temp_dir in ['temp_frames', 'temp_audio']:
    temp_path = os.path.join(cfg.TRAIN_DIR, temp_dir)
    if os.path.exists(temp_path):
        shutil.rmtree(temp_path)

Preprocessing videos:   0%|          | 0/400 [00:00<?, ?it/s]

Starting DFDC data preprocessing...
Input directory: /kaggle/input/competitions/deepfake-detection-challenge
Output directory: /kaggle/working


Preprocessing videos: 100%|██████████| 400/400 [09:56<00:00,  1.49s/it]


Total processed: 77 real, 300 fake videos


---
## 9. Generate CSV Split Files

In [14]:
def write_csv_list(data_list, path):
    with open(path, 'w', newline='') as f:
        writer = csv.writer(f, delimiter=',')
        for row in data_list:
            if row:
                writer.writerow(row)
    print(f'Split saved to {path} ({len(data_list)} entries)')


def generate_splits(work_dir, train_test_ratio=0.8):
    pytmp_real = os.path.join(work_dir, 'train', 'pytmp', 'real')
    pytmp_fake = os.path.join(work_dir, 'train', 'pytmp', 'fake')
    
    real_videos = []
    fake_videos = []
    
    if os.path.exists(pytmp_real):
        real_videos = [d for d in os.listdir(pytmp_real)
                       if os.path.isdir(os.path.join(pytmp_real, d))]
    if os.path.exists(pytmp_fake):
        fake_videos = [d for d in os.listdir(pytmp_fake)
                       if os.path.isdir(os.path.join(pytmp_fake, d))]
    
    print(f'Found {len(real_videos)} real videos, {len(fake_videos)} fake videos')
    
    random.shuffle(real_videos)
    random.shuffle(fake_videos)
    
    real_train_count = int(len(real_videos) * train_test_ratio)
    fake_train_count = int(len(fake_videos) * train_test_ratio)
    
    real_train = real_videos[:real_train_count]
    real_test = real_videos[real_train_count:]
    fake_train = fake_videos[:fake_train_count]
    fake_test = fake_videos[fake_train_count:]
    
    print(f'Train: {len(real_train)} real, {len(fake_train)} fake')
    print(f'Test: {len(real_test)} real, {len(fake_test)} fake')
    
    train_set = []
    base_train = os.path.join('train', 'pytmp')
    
    for video_name in fake_train:
        video_dir = os.path.join(pytmp_fake, video_name)
        for chunk in os.listdir(video_dir):
            chunk_path = os.path.join(video_dir, chunk)
            if os.path.isdir(chunk_path):
                audio_file = chunk + '.wav'
                audio_path = os.path.join(video_dir, audio_file)
                if os.path.exists(audio_path):
                    train_set.append([
                        os.path.join(base_train, 'fake', video_name, chunk),
                        os.path.join(base_train, 'fake', video_name, audio_file),
                        'fake'
                    ])
    
    for video_name in real_train:
        video_dir = os.path.join(pytmp_real, video_name)
        for chunk in os.listdir(video_dir):
            chunk_path = os.path.join(video_dir, chunk)
            if os.path.isdir(chunk_path):
                audio_file = chunk + '.wav'
                audio_path = os.path.join(video_dir, audio_file)
                if os.path.exists(audio_path):
                    train_set.append([
                        os.path.join(base_train, 'real', video_name, chunk),
                        os.path.join(base_train, 'real', video_name, audio_file),
                        'real'
                    ])
    
    test_set = []
    for video_name in fake_test:
        video_dir = os.path.join(pytmp_fake, video_name)
        for chunk in os.listdir(video_dir):
            chunk_path = os.path.join(video_dir, chunk)
            if os.path.isdir(chunk_path):
                audio_file = chunk + '.wav'
                audio_path = os.path.join(video_dir, audio_file)
                if os.path.exists(audio_path):
                    test_set.append([
                        os.path.join(base_train, 'fake', video_name, chunk),
                        os.path.join(base_train, 'fake', video_name, audio_file),
                        'fake'
                    ])
    
    for video_name in real_test:
        video_dir = os.path.join(pytmp_real, video_name)
        for chunk in os.listdir(video_dir):
            chunk_path = os.path.join(video_dir, chunk)
            if os.path.isdir(chunk_path):
                audio_file = chunk + '.wav'
                audio_path = os.path.join(video_dir, audio_file)
                if os.path.exists(audio_path):
                    test_set.append([
                        os.path.join(base_train, 'real', video_name, chunk),
                        os.path.join(base_train, 'real', video_name, audio_file),
                        'real'
                    ])
    
    random.shuffle(train_set)
    random.shuffle(test_set)
    
    write_csv_list(train_set, os.path.join(work_dir, 'train_split.csv'))
    write_csv_list(test_set, os.path.join(work_dir, 'test_imbalance_split.csv'))
    
    return len(train_set), len(test_set)


n_train, n_test = generate_splits(cfg.WORK_DIR)
print(f'\nTrain samples: {n_train}, Test samples: {n_test}')

Found 77 real videos, 300 fake videos
Train: 61 real, 240 fake
Test: 16 real, 60 fake
Split saved to /kaggle/working/train_split.csv (1505 entries)
Split saved to /kaggle/working/test_imbalance_split.csv (380 entries)

Train samples: 1505, Test samples: 380


---
## 10. Data Loaders

In [15]:
def get_rawaudio_data(transform, cfg, mode='train'):
    print(f'Loading data for "{mode}"...')
    dataset = deepfake_3d_rawaudio(
        out_dir=cfg.WORK_DIR, mode=mode, transform=transform,
        vis_min_fake_len=cfg.VIS_MIN_FAKE_LEN, vis_max_fake_len=cfg.VIS_MAX_FAKE_LEN,
        aud_min_fake_len=cfg.AUD_MIN_FAKE_LEN, aud_max_fake_len=cfg.AUD_MAX_FAKE_LEN,
        using_pseudo_fake=cfg.USING_PSEUDO_FAKE if mode == 'train' else False,
        dataset_name='dfdc'
    )
    sampler = data.RandomSampler(dataset)

    if mode == 'train':
        data_loader = data.DataLoader(
            dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler, shuffle=False,
            num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True,
            collate_fn=my_collate_rawaudio
        )
    elif mode == 'test':
        data_loader = data.DataLoader(
            dataset, batch_size=1, sampler=sampler, shuffle=False,
            num_workers=cfg.NUM_WORKERS, pin_memory=True,
            collate_fn=my_collate_rawaudio
        )
    else:
        raise ValueError(f'No mode {mode}')

    print(f'"{mode}" dataset size: {len(dataset)}')
    return data_loader


train_transform = transforms.Compose([
    Scale(size=(cfg.IMG_DIM, cfg.IMG_DIM)),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    Scale(size=(cfg.IMG_DIM, cfg.IMG_DIM)),
    ToTensor(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_loader = get_rawaudio_data(train_transform, cfg, 'train')
print(f'Train loader: {len(train_loader)} batches')

Loading data for "train"...
"train" dataset size: 1505
Train loader: 94 batches


---
## 11. Training Loop

In [16]:
def train_one_epoch(data_loader, model, optimizer, criterion, epoch, cfg, writer=None):
    losses = AverageMeter()
    real_distances = AverageMeter()
    fake_distances = AverageMeter()
    model.train()
    cuda_dev = torch.device('cuda')

    for idx, batch_data in enumerate(data_loader):
        if len(batch_data[0]) == 0:
            continue
        video_seq, audio_seq, target, audiopath = batch_data
        target = 1 - target

        tic = time.time()
        video_seq = video_seq.to(cuda_dev)
        audio_seq = audio_seq.to(cuda_dev)
        target = target.to(cuda_dev)
        B = video_seq.size(0)

        out, vid_out_dist, att = model(video_seq, audio_seq)
        del video_seq, audio_seq

        loss = criterion(out, target.float())
        losses.update(loss.item(), B)

        n_real = torch.sum(target.view(-1) == 0).detach().cpu().item()
        if n_real > 0:
            real_distances.update(torch.mean(vid_out_dist[target.view(-1) == 0]).item())
        n_fake = torch.sum(target.view(-1) == 1).detach().cpu().item()
        if n_fake > 0:
            fake_distances.update(torch.mean(vid_out_dist[target.view(-1) == 1]).item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if idx % cfg.PRINT_FREQ == 0:
            elapsed = time.time() - tic
            print(f'Epoch [{epoch}][{idx}/{len(data_loader)}] '
                  f'Time: {elapsed:.2f}s  '
                  f'Loss: {losses.val:.4f} ({losses.local_avg:.4f})  '
                  f'Real: {real_distances.val:.4f}  Fake: {fake_distances.val:.4f}')

        if writer:
            writer.add_scalar('local/loss', losses.val, epoch * len(data_loader) + idx)

    return losses.avg, real_distances.avg, fake_distances.avg


def test_model(data_loader, model, cuda_device):
    real_distances = AverageMeter()
    fake_distances = AverageMeter()
    model.eval()
    test_pred = {}
    test_target = {}
    test_num_chunks = {}

    with torch.no_grad():
        for idx, batch_data in tqdm(enumerate(data_loader), total=len(data_loader), desc='Testing'):
            if len(batch_data[0]) == 0:
                continue
            video_seq, audio_seq, target, audiopath = batch_data
            target = 1 - target

            video_seq = video_seq[0].unsqueeze(0).to(cuda_device)
            audio_seq = audio_seq[0].unsqueeze(0).to(cuda_device)
            target = target[0].unsqueeze(0).to(cuda_device)

            pred, vid_aud_dist, att = model(video_seq, audio_seq)
            del video_seq, audio_seq

            tar = target[0, :].view(-1).item()
            vid_name = audiopath[0].split('/')[-2]

            if test_pred.get(vid_name):
                test_pred[vid_name] += pred[0].view(-1).item()
                test_num_chunks[vid_name] += 1
            else:
                test_pred[vid_name] = pred[0].view(-1).item()
                test_num_chunks[vid_name] = 1

            if not test_target.get(vid_name):
                test_target[vid_name] = tar

            if tar == 1:
                fake_distances.update(torch.mean(vid_aud_dist[0]).item())
            else:
                real_distances.update(torch.mean(vid_aud_dist[0]).item())

    pred_fake = []
    pred_real = []
    for video, score in test_pred.items():
        tar = test_target[video]
        num_chunks = test_num_chunks[video]
        mean_pred = score / num_chunks
        if tar == 1:
            pred_fake.append(mean_pred)
        else:
            pred_real.append(mean_pred)

    if len(pred_fake) > 0 and len(pred_real) > 0:
        pred_all = np.concatenate([np.array(pred_fake), np.array(pred_real)])
        gt_all = np.concatenate([np.ones(len(pred_fake)), np.zeros(len(pred_real))])
        auc = roc_auc_score(gt_all, pred_all)
    else:
        auc = 0.0

    print(f'Fake dist: {fake_distances.avg:.4f}  Real dist: {real_distances.avg:.4f}  AUC: {auc:.4f}')
    return auc, real_distances.avg, fake_distances.avg

print('Training and testing functions defined.')

Training and testing functions defined.


In [17]:
# ====================== INITIALIZE MODEL ======================

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
cuda = torch.device('cuda')

model = My_Network(
    with_attention=cfg.WITH_ATTENTION,
    residual_conn=cfg.RESIDUAL_CONN,
    spatial_size=cfg.SPATIAL_SIZE
)
model = model.cuda()
model = nn.DataParallel(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)

log_dir = os.path.join(cfg.WORK_DIR, 'logs')
model_dir = os.path.join(cfg.WORK_DIR, 'models')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)
writer = SummaryWriter(logdir=log_dir)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable_params:,} / Total: {total_params:,} parameters')

Trainable: 3,604,177 / Total: 3,604,177 parameters


In [18]:
# ====================== MAIN TRAINING LOOP ======================

print('=' * 60)
print('Starting Training')
print(f'  Epochs: {cfg.EPOCHS}, BS: {cfg.BATCH_SIZE}, LR: {cfg.LEARNING_RATE}')
print(f'  Attention: {cfg.WITH_ATTENTION}, Pseudo-Fake: {cfg.USING_PSEUDO_FAKE}')
print('=' * 60)

least_loss = float('inf')
train_losses = []
train_real_dists = []
train_fake_dists = []

torch.backends.cudnn.benchmark = True

for epoch in range(cfg.EPOCHS):
    print(f'\n--- Epoch {epoch+1}/{cfg.EPOCHS} ---')
    
    train_loss, real_distance, fake_distance = train_one_epoch(
        train_loader, model, optimizer, criterion, epoch, cfg, writer
    )
    
    train_losses.append(train_loss)
    train_real_dists.append(real_distance)
    train_fake_dists.append(fake_distance)
    
    writer.add_scalar('global/loss', train_loss, epoch)
    writer.add_scalar('global/real_distance', real_distance, epoch)
    writer.add_scalar('global/fake_distance', fake_distance, epoch)
    
    print(f'Epoch {epoch+1}: Loss={train_loss:.4f}, '
          f'Real={real_distance:.4f}, Fake={fake_distance:.4f}')
    
    is_best = train_loss <= least_loss
    least_loss = min(least_loss, train_loss)
    
    save_checkpoint({
        'epoch': epoch + 1,
        'net': cfg.NET,
        'state_dict': model.state_dict(),
        'least_loss': least_loss,
        'optimizer': optimizer.state_dict(),
        'iteration': epoch
    }, is_best, filename=os.path.join(model_dir, f'epoch{epoch+1}.pth.tar'),
       keep_all=cfg.SAVE_ALL)
    
    if is_best:
        print(f'  * New best model! (loss={train_loss:.4f})')

print(f'\nTraining finished! Best loss: {least_loss:.4f}')
writer.close()

Starting Training
  Epochs: 100, BS: 16, LR: 0.0001
  Attention: True, Pseudo-Fake: True

--- Epoch 1/100 ---
Epoch [0][0/94] Time: 8.83s  Loss: 0.6831 (0.6831)  Real: 0.0000  Fake: 0.0774
Epoch [0][10/94] Time: 0.93s  Loss: 0.6407 (0.6555)  Real: 0.0763  Fake: 0.0841
Epoch [0][20/94] Time: 0.93s  Loss: 0.6392 (0.6064)  Real: 0.1033  Fake: 0.0943
Epoch [0][30/94] Time: 1.01s  Loss: 0.3631 (0.4936)  Real: 0.2633  Fake: 0.1905
Epoch [0][40/94] Time: 1.00s  Loss: 0.4901 (0.4835)  Real: 0.1904  Fake: 0.1909
Epoch [0][50/94] Time: 1.03s  Loss: 0.2357 (0.2856)  Real: 0.1253  Fake: 0.2291
Epoch [0][60/94] Time: 1.07s  Loss: 0.2779 (0.2145)  Real: 0.4181  Fake: 0.3500
Epoch [0][70/94] Time: 1.12s  Loss: 0.2815 (0.3466)  Real: 0.5370  Fake: 0.3838
Epoch [0][80/94] Time: 1.12s  Loss: 0.2242 (0.3804)  Real: 0.2659  Fake: 0.3043
Epoch [0][90/94] Time: 1.13s  Loss: 0.1309 (0.2703)  Real: 0.3399  Fake: 0.3337
Epoch 1: Loss=0.4422, Real=0.2225, Fake=0.2247
  * New best model! (loss=0.4422)

--- Epoch

---
## 12. Training Visualization

In [19]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_real_dists, 'g-', linewidth=2, label='Real')
axes[1].plot(train_fake_dists, 'r-', linewidth=2, label='Fake')
axes[1].set_title('Audio-Visual Distance', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Distance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

gap = [f - r for f, r in zip(train_fake_dists, train_real_dists)]
axes[2].plot(gap, 'm-', linewidth=2)
axes[2].set_title('Distance Gap (Fake - Real)', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gap')
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.WORK_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

Training curves saved.


---
## 13. Testing and Evaluation

In [20]:
# Load best model
best_model_path = glob.glob(os.path.join(model_dir, 'model_best_*.pth.tar'))
if best_model_path:
    best_model_path = best_model_path[0]
    print(f'Loading best model: {best_model_path}')
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['state_dict'])
    print(f'Loaded model from epoch {checkpoint["epoch"]}')
else:
    print('No best model found, using latest model state.')

# Create test loader
test_cfg = copy.deepcopy(cfg)
test_cfg.USING_PSEUDO_FAKE = False

test_loader = get_rawaudio_data(test_transform, test_cfg, 'test')
print(f'Test loader: {len(test_loader)} samples')

print('\n--- Running Evaluation ---')
auc, real_dist, fake_dist = test_model(test_loader, model, cuda)

print('\n' + '=' * 60)
print(f'  FINAL RESULTS')
print(f'  AUC Score: {auc:.4f}')
print(f'  Real Distance: {real_dist:.4f}')
print(f'  Fake Distance: {fake_dist:.4f}')
print('=' * 60)

Loading best model: /kaggle/working/models/model_best_epoch94.pth.tar
Loaded model from epoch 94
Loading data for "test"...
"test" dataset size: 380
Test loader: 380 samples

--- Running Evaluation ---


Testing: 100%|██████████| 380/380 [00:20<00:00, 18.27it/s]

Fake dist: 0.3107  Real dist: 0.3449  AUC: 0.5219

  FINAL RESULTS
  AUC Score: 0.5219
  Real Distance: 0.3449
  Fake Distance: 0.3107


---
## 14. Save Final Model

In [21]:
final_model_path = os.path.join(cfg.WORK_DIR, 'fgi_deepfake_detector_final.pth')
torch.save({
    'state_dict': model.state_dict(),
    'config': {
        'with_attention': cfg.WITH_ATTENTION,
        'residual_conn': cfg.RESIDUAL_CONN,
        'spatial_size': cfg.SPATIAL_SIZE,
        'net': cfg.NET,
    },
    'auc': auc,
}, final_model_path)

file_size = os.path.getsize(final_model_path) / (1024 * 1024)
print(f'Final model saved: {final_model_path} ({file_size:.2f} MB)')
print(f'AUC: {auc:.4f}')

Final model saved: /kaggle/working/fgi_deepfake_detector_final.pth (13.77 MB)
AUC: 0.5219


---
## 15. Inference on New Videos

In [22]:
def predict_video(video_path, model, transform, cuda_device, max_chunks=10):
    """
    Run deepfake prediction on a single video.
    Returns probability of being fake (closer to 1 = FAKE, closer to 0 = REAL).
    """
    model.eval()
    temp_dir = '/kaggle/working/temp_inference'
    os.makedirs(temp_dir, exist_ok=True)
    
    frames_dir = os.path.join(temp_dir, 'frames')
    os.makedirs(frames_dir, exist_ok=True)
    cmd = f'ffmpeg -y -i "{video_path}" -r 30 -qscale:v 2 -vf scale=224:224 -f image2 "{os.path.join(frames_dir, "%06d.jpg")}"'
    subprocess.call(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    audio_path = os.path.join(temp_dir, 'audio.wav')
    cmd = f'ffmpeg -y -i "{video_path}" -ac 1 -vn -acodec pcm_s16le -ar 48000 "{audio_path}"'
    subprocess.call(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    if not os.path.exists(audio_path):
        shutil.rmtree(temp_dir, ignore_errors=True)
        return None
    
    frames = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
    total_frames = len(frames)
    sample_rate, full_audio = wavfile.read(audio_path)
    
    predictions = []
    with torch.no_grad():
        for chunk_start in range(0, total_frames, 30):
            if chunk_start + 30 > total_frames:
                break
            if max_chunks and len(predictions) >= max_chunks:
                break
            
            chunk_frames = [pil_loader(os.path.join(frames_dir, frames[i]))
                           for i in range(chunk_start, chunk_start + 30)]
            t_seq = transform(chunk_frames)
            (C, H, W) = t_seq[0].size()
            t_seq = torch.stack(t_seq, 0)
            t_seq = t_seq.view(1, 1, 30, C, H, W).transpose(2, 3)
            
            audio_start = int(chunk_start / 30.0 * sample_rate)
            audio_end = int((chunk_start + 30) / 30.0 * sample_rate)
            chunk_audio = full_audio[audio_start:audio_end]
            
            if len(chunk_audio) < 48000:
                chunk_audio = np.pad(chunk_audio, (0, 48000 - len(chunk_audio)))
            elif len(chunk_audio) > 48000:
                chunk_audio = chunk_audio[:48000]
            
            norm_audio = min_max_normalize(chunk_audio, int(chunk_audio.min()), int(chunk_audio.max()))
            norm_audio = torch.from_numpy(norm_audio.astype(float)).float()
            norm_audio = (norm_audio - 0.5) / 0.5
            norm_audio = norm_audio.unsqueeze(0)
            
            pred, _, _ = model(t_seq.to(cuda_device), norm_audio.to(cuda_device))
            predictions.append(pred[0].item())
    
    shutil.rmtree(temp_dir, ignore_errors=True)
    return np.mean(predictions) if predictions else None


print('Inference function defined.')
print('Usage: score = predict_video("/path/to/video.mp4", model, test_transform, cuda)')
print('  score close to 1 = FAKE, close to 0 = REAL')

Inference function defined.
Usage: score = predict_video("/path/to/video.mp4", model, test_transform, cuda)
  score close to 1 = FAKE, close to 0 = REAL


In [23]:
# Example inference (uncomment to test):

# sample_video = '/kaggle/input/deepfake-detection-challenge/train_sample_videos/aaqaifqrwn.mp4'
# if os.path.exists(sample_video):
#     score = predict_video(sample_video, model, test_transform, cuda)
#     if score is not None:
#         label = 'FAKE' if score > 0.5 else 'REAL'
#         print(f'Video: {os.path.basename(sample_video)}')
#         print(f'Score: {score:.4f} -> Prediction: {label}')

print('\nNotebook complete!')
print('To train on full DFDC dataset, set MAX_VIDEOS = None in the preprocessing cell.')


Notebook complete!
To train on full DFDC dataset, set MAX_VIDEOS = None in the preprocessing cell.
